# CNN Classifier — MobileNetV3 + Custom CNN para Autenticidad

Compara 3 arquitecturas convolucionales entrenadas con 1433 falsos reales + 1433 autenticos.
Todas exportan a TFLite para RPi Zero 2W.

In [ ]:
import os, random, numpy as np, cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 2. Configuracion

In [ ]:
AUTH_DIR = Path("../rpi_deployment/training_data/autenticos")
FAKES_DIR = Path("../Data/raw/fakes")
IMG_SIZE = 96  # mobile-optimizado
BATCH_SIZE = 32
EPOCHS = 25
TRAIN_SPLIT = 0.8

print(f"Input: {IMG_SIZE}x{IMG_SIZE}")

## 3. Cargar datos balanceados (padding, no stretch)

In [ ]:
def load_and_pad(path, max_samples=None):
    """Carga imagen, padding a cuadrado preservando aspect ratio."""
    images = []
    for f in path:
        img = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        h, w = img.shape[:2]
        scale = IMG_SIZE / max(h, w)
        nh, nw = int(h * scale), int(w * scale)
        resized = cv2.resize(img, (nw, nh))
        canvas = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.float32)
        yo, xo = (IMG_SIZE - nh) // 2, (IMG_SIZE - nw) // 2
        canvas[yo:yo+nh, xo:xo+nw] = resized.astype(np.float32) / 255.0
        images.append(canvas)
        if max_samples and len(images) >= max_samples: break
    return np.array(images, dtype=np.float32)

# Autenticos
auth_files = list(AUTH_DIR.glob("*.png"))
random.shuffle(auth_files)
X_auth = load_and_pad(auth_files)
print(f"Autenticos: {len(X_auth)}")

# Falsos
fake_files = list(FAKES_DIR.rglob("*.jpg"))
random.shuffle(fake_files)
X_fake = load_and_pad(fake_files)
print(f"Falsos: {len(X_fake)}")

# Balancear
n = min(len(X_auth), len(X_fake))
X_auth, X_fake = X_auth[:n], X_fake[:n]

# Concatenar y agregar canal
X = np.concatenate([X_auth, X_fake]).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.hstack([np.ones(n), np.zeros(n)])

# Shuffle + split
idx = np.random.permutation(len(X))
split = int(TRAIN_SPLIT * len(X))
X_train, y_train = X[idx[:split]], y[idx[:split]]
X_val, y_val = X[idx[split:]], y[idx[split:]]

print(f"Train: {len(X_train)}, Val: {len(X_val)}")

## 4. MobileNetV3-Small (Transfer Learning)

In [ ]:
base = applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 1),
    include_top=False,
    weights=None,  # sin ImageNet (grayscale)
    alpha=1.0,
    minimalistic=True
)
base.trainable = True

mobilenet = keras.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])
mobilenet.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
mobilenet.summary()

In [ ]:
h1 = mobilenet.fit(X_train, y_train, validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
loss1, acc1 = mobilenet.evaluate(X_val, y_val, verbose=0)
print(f"MobileNetV3: Val Acc = {acc1*100:.2f}%")

## 5. Custom CNN (ligera, desde cero)

In [ ]:
custom_cnn = keras.Sequential([
    layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(IMG_SIZE,IMG_SIZE,1)),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Conv2D(256, 3, activation='relu', padding='same'),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])
custom_cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
custom_cnn.summary()

In [ ]:
h2 = custom_cnn.fit(X_train, y_train, validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
loss2, acc2 = custom_cnn.evaluate(X_val, y_val, verbose=0)
print(f"Custom CNN: Val Acc = {acc2*100:.2f}%")

## 6. EfficientNet-B0 (mejor accuracy/size)

In [ ]:
base3 = applications.EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 1),
    include_top=False,
    weights=None
)
base3.trainable = True

effnet = keras.Sequential([
    base3,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])
effnet.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy'])
effnet.summary()

In [ ]:
h3 = effnet.fit(X_train, y_train, validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
loss3, acc3 = effnet.evaluate(X_val, y_val, verbose=0)
print(f"EfficientNetB0: Val Acc = {acc3*100:.2f}%")

## 7. Comparativa final

In [ ]:
print(f"{'Modelo':25s} {'Acc':>8s} {'Params':>10s}")
print("-" * 45)
for name, acc, model in [
    ("Random Forest (baseline)", 0.9913, None),
    ("MobileNetV3-Small", acc1, mobilenet),
    ("Custom CNN (4-layer)", acc2, custom_cnn),
    ("EfficientNet-B0", acc3, effnet),
]:
    params = model.count_params() if model else 0
    print(f"{name:25s} {acc*100:8.2f}% {params:10,d}")

# Grafico comparativo
models = ['RF (base)', 'MobileNetV3', 'Custom CNN', 'EfficientNetB0']
accs = [99.13, acc1*100, acc2*100, acc3*100]
colors = ['gray', 'blue', 'green', 'orange']
plt.figure(figsize=(8, 5))
bars = plt.bar(models, accs, color=colors)
plt.axhline(y=99.13, color='gray', linestyle='--', label='RF baseline (99.13%)')
plt.ylabel('Accuracy (%)'); plt.legend()
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{acc:.1f}%', ha='center', fontweight='bold')
plt.title('CNN Models vs Random Forest')
plt.tight_layout(); plt.show()

## 8. Exportar el mejor a TFLite + ONNX

In [ ]:
# Elegir el mejor
best = max([(acc1, mobilenet, 'mobilenet'), (acc2, custom_cnn, 'custom'), (acc3, effnet, 'effnet')])
best_model = best[1]
print(f"Mejor: {best[2]} ({best[0]*100:.2f}%)")

# TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite = converter.convert()
tflite_path = f"../Models_rpi/cnn_classifier_{best[2]}.tflite"
with open(tflite_path, 'wb') as f: f.write(tflite)
print(f"TFLite: {tflite_path} ({len(tflite)/1024:.1f} KB)")